# Демо: threading и multiprocessing на практике

Прокликай Shift+Enter каждую ячейку и посмотри: race condition вживую и как `Lock` его чинит, как `queue.Queue` решает producer/consumer, и как `concurrent.futures` объединяет всё в одном API. В конце — три мини-задания.

**Как читать этот ноутбук.** Часть примеров — обычные ячейки, запускаются здесь по `Shift+Enter`. Часть (всё про `multiprocessing.Pool`) вынесена в `.py`-скрипты в папке `scripts/`. В таких местах ты увидишь блок:

> **→ Перейди в терминал.** `python scripts/<имя>.py`

Открой терминал в папке `notebooks/`, запусти указанный скрипт, сравни вывод с тем, что в ноутбуке, и возвращайся к следующей ячейке. Причина — `multiprocessing.Pool` в Jupyter на Windows не работает (дочерние процессы не видят функции, объявленные в ячейках). Поэтому правильный паттерн для `Pool` — отдельный скрипт с гвардом `if __name__ == "__main__":`. Подробнее в `scripts/README.md`.

## Часть 1. Race condition: один инкремент теряется

Запустим 5 потоков, каждый сделает 200_000 раз `counter += 1`. Без синхронизации финальное значение **обычно меньше** ожидаемого 1_000_000 — некоторые инкременты теряются между потоками.

Почему теряются: `counter += 1` это три байткода (read, add, write). Поток может прочитать `42`, GIL переключился, второй поток тоже прочитал `42`, оба прибавили, оба записали `43` — один инкремент потерян.

In [1]:
import threading

counter = 0

def worker_unsafe(n):
    global counter
    for _ in range(n):
        counter += 1   # НЕ-атомарная операция

threads = [threading.Thread(target=worker_unsafe, args=(200_000,)) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"ожидалось:    1_000_000")
print(f"получили:     {counter}")
print(f"потерялось:   {1_000_000 - counter}")


ожидалось:    1_000_000
получили:     922810
потерялось:   77190


Заметь: число «получили» меняется от запуска к запуску. Это и пугает в race condition — баг проявляется не каждый раз, его сложно воспроизвести в тестах.

## Часть 2. Починка через `Lock`

Решение — `threading.Lock`. Конструкция `with lock:` гарантирует, что в каждый момент только один поток внутри критической секции. Остальные ждут на `acquire()`.

In [2]:
import threading

counter = 0
lock = threading.Lock()

def worker_safe(n):
    global counter
    for _ in range(n):
        with lock:
            counter += 1

threads = [threading.Thread(target=worker_safe, args=(200_000,)) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"ожидалось:  1_000_000")
print(f"получили:   {counter}")
print(f"совпало:    {counter == 1_000_000}")


ожидалось:  1_000_000
получили:   1000000
совпало:    True


Цена `Lock` — потоки сериализуются на критической секции, часть преимущества от потоков пропадает. Поэтому держим `with lock:` узкой и оборачиваем только то, что реально нужно защитить.

## Часть 3. `queue.Queue` — producer/consumer без `Lock`

`queue.Queue` — потокобезопасная очередь. Внутри уже есть `Lock`, поэтому `put`/`get` атомарны. Это удобный паттерн для producer/consumer: один поток кладёт работу, другой забирает.

In [3]:
import threading
import queue
import time

tasks: queue.Queue = queue.Queue()

def producer():
    for i in range(5):
        print(f"  producer кладёт {i}")
        tasks.put(i)
        time.sleep(0.05)
    tasks.put(None)   # sentinel

def consumer():
    while True:
        item = tasks.get()
        if item is None:
            tasks.task_done()
            break
        print(f"     consumer обработал {item} → {item ** 2}")
        tasks.task_done()

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
p.start()
c.start()
p.join()
c.join()
print("очередь обработана")


  producer кладёт 0
     consumer обработал 0 → 0
  producer кладёт 1
     consumer обработал 1 → 1


  producer кладёт 2
     consumer обработал 2 → 4
  producer кладёт 3
     consumer обработал 3 → 9


  producer кладёт 4
     consumer обработал 4 → 16
очередь обработана


## Часть 4. `multiprocessing.Pool` — параллельная обработка списка

`Pool(N)` создаёт N процессов-воркеров. `pool.map(func, items)` распределяет элементы списка между воркерами и собирает результаты в порядке исходного списка. Идиома: «обработать миллион картинок» или «парсить тысячу JSON-логов».

> **→ Перейди в терминал.** Этот пример не запускается из ноутбука — только из `.py`-скрипта. Открой терминал в папке `notebooks/` и выполни:
>
> ```bash
> python scripts/demo_pool_map.py
> ```
>
> Когда увидишь вывод — возвращайся сюда и читай дальше.

Ожидаемый вывод (точные числа зависят от машины):

```
последовательно: 0.12s
Pool(4):         0.07s
speedup:         1.82x
результаты совпали: True
```

Замечание про сериализацию: всё, что передаётся в `pool.map`, должно быть **picklable**. Лямбды, локально-объявленные классы, открытые файловые дескрипторы — нет. Глобальные функции и стандартные контейнеры — да. На практике это означает: функция-воркер должна лежать на верхнем уровне модуля — что мы и видим в скрипте.

## Часть 5. `concurrent.futures` — единый интерфейс

Модуль `concurrent.futures` объединяет потоки и процессы под одним API. Меняешь один класс (`ThreadPoolExecutor` ↔ `ProcessPoolExecutor`) — и тот же код работает либо на потоках, либо на процессах. Удобно для бенчмарков и переключения стратегий.

In [4]:
from concurrent.futures import ThreadPoolExecutor
import time

def fake_io(name):
    time.sleep(0.2)
    return f"<{name}>"

names = ["alpha", "beta", "gamma", "delta"]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(fake_io, names))
elapsed = time.perf_counter() - start

print("результаты:", results)
print(f"время:      {elapsed:.2f}s — близко к 0.2 (max), не 0.8 (sum)")


результаты: ['<alpha>', '<beta>', '<gamma>', '<delta>']
время:      0.20s — близко к 0.2 (max), не 0.8 (sum)


Альтернатива: `submit` + итерация по списку futures — для случаев, когда хочешь явно держать ссылки на отдельные задачи и забирать их результаты в исходном порядке:

In [5]:
from concurrent.futures import ThreadPoolExecutor
import time

def fake_request(name, latency):
    time.sleep(latency)
    return name, latency

tasks = [("alpha", 0.3), ("beta", 0.1), ("gamma", 0.2)]

with ThreadPoolExecutor(max_workers=3) as ex:
    futures = [ex.submit(fake_request, n, lat) for n, lat in tasks]
    for fut in futures:
        name, lat = fut.result()
        print(f"  готов {name} (latency={lat}s)")
# порядок результатов — по исходному списку tasks


  готов alpha (latency=0.3s)
  готов beta (latency=0.1s)
  готов gamma (latency=0.2s)


## Часть 6. Что когда брать — шпаргалка

| Сценарий | Инструмент |
|---|---|
| 5 одновременных HTTP-запросов | `ThreadPoolExecutor.map` или `asyncio.gather` |
| Обработка 1М картинок в Pillow | `multiprocessing.Pool.map` |
| Producer/consumer с очередью | `queue.Queue` + 2 потока |
| Защита счётчика от гонки | `threading.Lock` |
| Параллельный numpy/pandas | `ThreadPoolExecutor` (NumPy сама отпускает GIL) |
| Подготовка батчей в DataLoader | `multiprocessing` (внутри `num_workers`) |
| Тысячи параллельных соединений | `asyncio` (следующий демо) |

## Мини-задания

Три коротких упражнения. Подсказок к именам и API нет — вспомни сам.

**Задание 1.** Реализуй потокобезопасный счётчик: класс `Counter` с методом `incr()`, использующий `threading.Lock`. Запусти 4 потока × 100_000 инкрементов. Финальное значение должно быть `400_000`.

**Задание 2.** Через `multiprocessing.Pool` запусти параллельную обработку списка `[1, 2, 3, 4, 5, 6, 7, 8]` функцией `def square(x): return x ** 2`. Подсказка: `lambda` не picklable, поэтому используем обычный `def`.

> **→ Перейди в терминал.** Это задание решается в отдельном `.py`-файле — заведи новый, например `scripts/my_square.py`, по образцу `scripts/demo_pool_map.py`, и запусти из папки `notebooks/`:
>
> ```bash
> python scripts/my_square.py
> ```
>
> Обязательный шаблон: функция `square` на верхнем уровне модуля + блок `if __name__ == "__main__":` вокруг `Pool`.

**Задание 3.** Что напечатает код ниже? Сначала угадай порядок и общее время, потом запусти.

In [5]:
# Задание 1
import threading
lock = threading.Lock()
class Counter:
    def __init__(self):
        self.counter = 0

    def incr(self, n):
        for i in range(n):
            with lock:
                self.counter += 1
c = Counter()
# 4 потока × 100_000 инкрементов
threads = [threading.Thread(target=c.incr, args=(100_000,)) for _ in range(4)]
for i in threads:
    i.start()
for i in threads:
    i.join()
print(f"ожидалось:  400_000")
print(f"получили:   {c.counter}")
print(f"совпало:    {c.counter == 400_000}")

ожидалось:  400_000
получили:   400000
совпало:    True


In [6]:
# Задание 3 — угадай порядок и общее время:
from concurrent.futures import ThreadPoolExecutor
import time

def task(name, delay):
    time.sleep(delay)
    return name

items = [("first",  0.4), ("second", 0.1), ("third", 0.2)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as ex:
    results = list(ex.map(lambda p: task(*p), items))
elapsed = time.perf_counter() - start

print("results:", results)            # first, second, third — в порядке исходного списка items
print(f"elapsed: {elapsed:.2f}s")    # 0.4


results: ['first', 'second', 'third']
elapsed: 0.40s
